In [0]:
from pyspark.sql import functions as F

BASE_PATH = "/Volumes/workspace/lakehouse_demo/raw_files"
SCHEMA = "workspace.lakehouse_demo"

FILES = {
    "customers": "customers_raw.csv",
    "products": "products_raw.csv",
    "orders": "orders_raw.csv",
}

EXPECTED_COLUMNS = {
    "customers": [
        "customer_id", "first_name", "last_name", "email",
        "phone_number", "gender", "dob", "signup_date",
        "address", "city", "state", "country", "device_id(s)", "source"
    ],
    "products": [
        "product_id", "product_name", "category", "price"
    ],
    "orders": [
        "order_id", "customer_id", "product_id", "order_amount",
        "order_date", "payment_method", "status", "quantity"
    ],
}

In [0]:
raw_data = {}

for entity, filename in FILES.items():
    df = (
        spark.read
        .option("header", "true")
        .option("sep", ",")
        .option("encoding", "UTF-8")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .csv(f"{BASE_PATH}/{filename}")
    )

    if df.columns != EXPECTED_COLUMNS[entity]:
        raise ValueError(
            f"Colonnes inattendues dans {filename}.\n"
            f"Attendues : {EXPECTED_COLUMNS[entity]}\n"
            f"Trouvées : {df.columns}"
        )

    raw_data[entity] = df

    print(f"{filename} — colonnes vérifiées")
    display(df.limit(5))

customers_raw.csv — colonnes vérifiées


customer_id,first_name,last_name,email,phone_number,gender,dob,signup_date,address,city,state,country,device_id(s),source
0d244537-8164-4b84-bcf2-0e43766f3221,mARIA,day,maria.day697@hotmail.com,449.977.1729x282,M,1988-04-11,2016-12-05,107 Joseph Station,Padillaborough,Oregon,Micronesia,56f3a452-b3f5-488b-b90d-8cb2b3343c70;3ab7d21b-48dd-44f8-88ae-c99a4dd51ef1;faced7cc-4d83-4505-9d86-497626fe16d0;dd49cd20-eacc-49f1-b4b5-92551e77d916,referral
66c5800b-ad5c-4fac-86df-16472d82f02d,KKeevvin,Cantu,kevin.cantu968@gmail.com,501.347.4824x64633,M,1955-09-19,2024-02-03,3577 Anita Knoll Apt. 107,Justinton,Delaware,Portugal,cd83e240-0d50-4e67-b146-23deb6c49940,referral
e7852eee-0a28-46dc-aa34-445a02530746,Ronald,Daniel,ronald.daniel138@yahoo.com,+1-268-822-6332x26814,M,1971-11-02,2016-04-12,3440 Tammy Views Suite 698,New Autumnland,Mississippi,Uganda,369b8950-dfae-44ce-bbb4-d292f118be67,web
14f0e67d-e60e-4989-88cf-5be50d515a69,Katherine,Smith,katherine.smith357@yahoo.com,864-894-1347,F,1955-03-25,2019-06-21,98888 Morales Lakes,New Jose,Minnesota,Bahrain,3e83071f-53ed-4a05-9dfd-5ff2031443b5,web
1ce2f061-345c-4121-a4ba-32963cf9a15d,Brandon,Glover,brandon.glover804@hotmail.com,489.732.5436,M,1968-06-25,2022-01-09,55037 Denise Crest Apt. 565,Port Amber,Oregon,Latvia,751d4410-3d73-4f4c-b432-10a0213b925f,referral


products_raw.csv — colonnes vérifiées


product_id,product_name,category,price
PROD-0001,Open Beaautiful,clo,32309.95
PROD-0002,Race Third,automotive,47346.82
PROD-0003,MEAN YOUR,3l3ctronics,-100
PROD-0004,Consumer Newspaper,toys,28506.95
PROD-0005,A Hot,clothing,19165.58


orders_raw.csv — colonnes vérifiées


order_id,customer_id,product_id,order_amount,order_date,payment_method,status,quantity
06b5aa88-f86f-4e07-8e4b-6df36402c209,0a331d4c-a73d-4271-83ac-2f6726635495,PROD-0060,473.82,2025-07-29,wall-et,success,1
5b10b856-e084-471c-bc16-8dbab95e3f33,9e48263f-7923-4cb9-8138-b9da8eb85619,PROD-0393,17796.32,2024/31/01,card,success,1
88706122-160e-4721-b537-73cee232fcd5,245fac1f-5ac1-4448-acc9-70ff980a4e9b,PROD-0332,31201.75,2024-03-02,wallet,success,2
c6ae75af-0313-4194-a788-8802515af2e4,abfda25e-6a38-4aa8-be24-cd22f371b2b0,PROD-0059,42282.89,null,wallet,refunded,2
032ab1c7-8882-47d8-a2ac-fbefc1cfb2eb,e0010492-b072-4f48-abd6-8a87938fad24,PROD-0342,27100.3,2024-02-11,wallet,success,1


In [0]:
for entity, df in raw_data.items():

    # Adapter le nom de colonne pour Delta
    if entity == "customers":
        df = df.withColumnRenamed("device_id(s)", "device_ids")

    bronze_df = (
        df
        .withColumn("_source_file", F.lit(FILES[entity]))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    table_name = f"{SCHEMA}.bronze_{entity}"

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print(f"Table créée : {table_name}")

Table créée : workspace.lakehouse_demo.bronze_customers
Table créée : workspace.lakehouse_demo.bronze_products
Table créée : workspace.lakehouse_demo.bronze_orders


In [0]:
for entity in FILES:
    table_name = f"{SCHEMA}.bronze_{entity}"
    df = spark.table(table_name)

    print(f"{table_name} : {df.count():,} lignes")
    df.printSchema()

workspace.lakehouse_demo.bronze_customers : 50,000 lignes
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device_ids: string (nullable = true)
 |-- source: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

workspace.lakehouse_demo.bronze_products : 500 lignes
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timest